# 04 -- Response Coherence

Measures how semantically coherent each reply is relative to its parent tweet
using three complementary methods:

| Method | Model | Score range |
|---|---|---|
| Cosine similarity | `sentence-transformers/all-MiniLM-L6-v2` | [0, 1] |
| BERTScore F1 | `roberta-large` | ~[0.7, 1.0] |
| Cross-encoder | `cross-encoder/stsb-roberta-base` | [0, 1] (sigmoid) |

A reply is flagged as incoherent when at least 2 of the 3 methods agree it falls
below its threshold (consensus flagging).

Two pairing strategies are available:
- **Thread-root pairing** -- every reply is compared to the original root tweet
- **Direct-parent pairing** -- every reply is compared to the tweet it directly replies to

**Input:** `DATA_DIR/tweets.csv` (output of `01_eda.ipynb`)
**Output:** `DATA_DIR/coherence_results.csv`, `DATA_DIR/coherence_summary.csv`, plots in `PLOT_DIR/`


## Dependencies

In [ ]:
# Run once -- safe to skip if already installed
# !pip install bert-score torch transformers sentence-transformers


## Imports

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
import pandas as pd
import seaborn as sns
import torch
from bert_score import score as bert_score
from scipy.stats import gaussian_kde
from sentence_transformers import CrossEncoder, SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity as cos_sim
from tqdm import tqdm


## Config

Set `DATA_DIR` to the folder produced by `01_eda.ipynb`.
All scoring thresholds can be tuned here without touching any other cell.


In [ ]:
DATA_DIR = "data"   # <- change to your local path
PLOT_DIR = f"{DATA_DIR}/plots"

INPUT_CSV       = f"{DATA_DIR}/tweets.csv"
OUTPUT_RESULTS  = f"{DATA_DIR}/coherence_results.csv"
OUTPUT_SUMMARY  = f"{DATA_DIR}/coherence_summary.csv"

# Models
EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
BERTSCORE_MODEL = "roberta-large"
CROSS_ENC_MODEL = "cross-encoder/stsb-roberta-base"

# Flagging thresholds
COSINE_LOW_THRESHOLD    = 0.25   # below this -> flagged
BERTSCORE_LOW_THRESHOLD = 0.85   # F1 below this -> flagged
CROSS_ENC_LOW_THRESHOLD = 0.50   # sigmoid score below this -> flagged

MAX_TEXT_LENGTH = 512
BATCH_SIZE      = 64    # reduce if you hit memory issues
DEVICE          = "cuda" if torch.cuda.is_available() else "cpu"

print(f"[config] Device          : {DEVICE}")
print(f"[config] Embedding model : {EMBEDDING_MODEL}")
print(f"[config] BERTScore model : {BERTSCORE_MODEL}")
print(f"[config] Cross-encoder   : {CROSS_ENC_MODEL}")


## Visualisation palette

Centralised colour constants and rcParams so all plots share a consistent style.


In [ ]:
C_TEAL    = "#1D9E75"
C_PURPLE  = "#7F77DD"
C_CORAL   = "#D85A30"
C_TEAL_L  = "#9FE1CB"
C_PURPLE_L= "#AFA9EC"
C_CORAL_L = "#F0997B"
C_RED     = "#E24B4A"
C_GRAY    = "#888780"
C_BG      = "#FAFAF8"
C_GRID    = "#EEECEA"

METRIC_COLORS = {
    "cosine_similarity":    (C_TEAL,   C_TEAL_L,   COSINE_LOW_THRESHOLD,    (0.0, 1.0)),
    "bs_f1":                (C_PURPLE, C_PURPLE_L, BERTSCORE_LOW_THRESHOLD, (0.7, 1.0)),
    "cross_encoder_score":  (C_CORAL,  C_CORAL_L,  CROSS_ENC_LOW_THRESHOLD, (0.0, 1.0)),
}
METRIC_LABELS = {
    "cosine_similarity":   "Cosine similarity",
    "bs_f1":               "BERTScore F1",
    "cross_encoder_score": "Cross-encoder",
}

plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "axes.facecolor": C_BG, "figure.facecolor": "white",
    "axes.edgecolor": "#D3D1C7", "axes.linewidth": 0.6,
    "axes.grid": True, "grid.color": C_GRID, "grid.linewidth": 0.5,
    "xtick.color": C_GRAY, "ytick.color": C_GRAY,
    "xtick.labelsize": 9, "ytick.labelsize": 9,
    "axes.labelsize": 10, "axes.titlesize": 11,
    "axes.titleweight": "bold", "axes.titlepad": 10,
    "legend.fontsize": 9, "legend.frameon": False,
})


## Data loading

Two loader variants are provided:

- `load_thread_root` -- pairs every reply with the **original root** of its thread.
  Use this to measure how far a conversation drifts from its starting point.
- `load_direct_parent` -- pairs every reply with its **immediate parent**.
  Use this to measure turn-by-turn coherence.

Both return the same `(roots, replies)` interface so the scoring functions below
work unchanged with either strategy.


In [ ]:
def load_thread_root(path: str) -> tuple:
    """
    Pair each reply with the original root tweet of its thread.

    Returns:
        roots   -- one row per thread_id (the root tweet)
        replies -- all non-root tweets merged with their thread root
    """
    df = pd.read_csv(path)
    print(f"[data] Loaded {len(df):,} rows, {df['thread_id'].nunique():,} unique threads")

    df["is_root"] = df["comment_to"] == -1
    if df["is_root"].sum() == 0:
        min_rounds = df.groupby("thread_id")["round"].transform("min")
        df["is_root"] = df["round"] == min_rounds

    roots = (
        df[df["is_root"]]
        .set_index("thread_id")[["id", "tweet"]]
        .rename(columns={"id": "root_id", "tweet": "root_tweet"})
    )
    replies = df[~df["is_root"] & df["thread_id"].notna()].copy()
    replies = replies.merge(roots, on="thread_id", how="inner")

    print(f"[data] Root tweets  : {len(roots):,}")
    print(f"[data] Reply tweets : {len(replies):,}")
    return roots, replies


def load_direct_parent(path: str) -> tuple:
    """
    Pair each reply with its direct parent tweet (not necessarily the thread root).

    Returns:
        roots   -- every tweet that received at least one reply
        replies -- all non-root tweets merged with their direct parent text
    """
    df = pd.read_csv(path)
    print(f"[data] Loaded {len(df):,} rows, {df['thread_id'].nunique():,} unique threads")

    df["is_root"] = df["comment_to"] == -1
    if df["is_root"].sum() == 0:
        min_rounds = df.groupby("thread_id")["round"].transform("min")
        df["is_root"] = df["round"] == min_rounds

    parent_ids = set(df.loc[df["comment_to"] != -1, "comment_to"])
    df["is_local_root"] = df["is_root"] | df["id"].isin(parent_ids)

    roots = (
        df[df["is_local_root"]]
        .set_index("id")[["thread_id", "tweet"]]
        .rename(columns={"tweet": "root_tweet"})
        .reset_index()
        .rename(columns={"id": "root_id"})
    )

    replies = df[~df["is_root"] & df["thread_id"].notna()].copy()
    replies = replies.merge(
        roots[["root_id", "root_tweet"]],
        left_on="comment_to", right_on="root_id", how="inner",
    )

    print(f"[data] Parent tweets : {len(roots):,}")
    print(f"[data] Reply tweets  : {len(replies):,}")
    return roots, replies


## Scoring functions

Each function takes the `replies` DataFrame (which must have `tweet` and `root_tweet` columns)
and returns a Series or DataFrame of scores aligned to the same index.


In [ ]:
def compute_cosine_similarity(replies: pd.DataFrame) -> pd.Series:
    """
    Encode root_tweet and reply tweet with a sentence transformer, then
    return the row-wise cosine similarity (vectors are L2-normalised so
    this reduces to a dot product).
    """
    print("\n[embedding] Loading model ...")
    model = SentenceTransformer(EMBEDDING_MODEL, device=DEVICE)

    print("[embedding] Encoding replies ...")
    emb_reply = model.encode(replies["tweet"].tolist(), batch_size=BATCH_SIZE,
                             show_progress_bar=True, normalize_embeddings=True)
    print("[embedding] Encoding roots ...")
    emb_root  = model.encode(replies["root_tweet"].tolist(), batch_size=BATCH_SIZE,
                             show_progress_bar=True, normalize_embeddings=True)

    scores = (emb_reply * emb_root).sum(axis=1)
    return pd.Series(scores, index=replies.index, name="cosine_similarity")


def compute_bertscore(replies: pd.DataFrame) -> pd.DataFrame:
    """
    BERTScore treating reply as hypothesis and root as reference.
    Returns a DataFrame with columns: bs_precision, bs_recall, bs_f1.
    """
    print("\n[bertscore] Computing BERTScore ...")
    P, R, F1 = bert_score(
        replies["tweet"].tolist(),
        replies["root_tweet"].tolist(),
        model_type=BERTSCORE_MODEL,
        lang="en",
        device=DEVICE,
        batch_size=max(1, BATCH_SIZE // 4),
        verbose=True,
    )
    return pd.DataFrame(
        {"bs_precision": P.numpy(), "bs_recall": R.numpy(), "bs_f1": F1.numpy()},
        index=replies.index,
    )


def compute_cross_encoder(replies: pd.DataFrame) -> pd.Series:
    """
    Cross-encoder relevance score for (root, reply) pairs.
    Raw logits are sigmoid-normalised to [0, 1].
    """
    print("\n[cross-encoder] Loading model ...")
    ce_model = CrossEncoder(CROSS_ENC_MODEL, device=DEVICE)

    pairs = list(zip(replies["root_tweet"], replies["tweet"]))
    print("[cross-encoder] Scoring pairs ...")
    raw = []
    for i in tqdm(range(0, len(pairs), BATCH_SIZE)):
        raw.extend(ce_model.predict(pairs[i : i + BATCH_SIZE]).tolist())

    scores = 1 / (1 + np.exp(-np.array(raw)))   # sigmoid
    return pd.Series(scores, index=replies.index, name="cross_encoder_score")


## Flagging and thread-level aggregation

`flag_low_coherence` adds boolean flag columns for each method and a
`flag_consensus` column that is `True` when at least 2 of the 3 methods
agree the reply is incoherent.

`summarise_by_thread` reduces reply-level scores to one row per thread,
including a composite coherence score that combines all three methods.


In [ ]:
def flag_low_coherence(df: pd.DataFrame) -> pd.DataFrame:
    """Add boolean flag columns per method and a consensus flag."""
    df = df.copy()
    df["flag_low_cosine"]    = df["cosine_similarity"]    < COSINE_LOW_THRESHOLD
    df["flag_low_bertscore"] = df["bs_f1"]                < BERTSCORE_LOW_THRESHOLD
    df["flag_low_crossenc"]  = df["cross_encoder_score"]  < CROSS_ENC_LOW_THRESHOLD
    df["flag_consensus"] = (
        df["flag_low_cosine"].astype(int) +
        df["flag_low_bertscore"].astype(int) +
        df["flag_low_crossenc"].astype(int)
    ) >= 2
    return df


def summarise_by_thread(df: pd.DataFrame) -> pd.DataFrame:
    """Per-thread aggregate statistics sorted by composite coherence (ascending)."""
    agg = df.groupby("thread_id").agg(
        n_replies             = ("id", "count"),
        mean_cosine           = ("cosine_similarity", "mean"),
        min_cosine            = ("cosine_similarity", "min"),
        mean_bertscore_f1     = ("bs_f1", "mean"),
        min_bertscore_f1      = ("bs_f1", "min"),
        mean_cross_encoder    = ("cross_encoder_score", "mean"),
        min_cross_encoder     = ("cross_encoder_score", "min"),
        pct_flagged_consensus = ("flag_consensus", "mean"),
    ).reset_index()

    # Composite: cosine [0,1] + rescaled BERTScore [0,1] + cross-encoder [0,1]
    agg["composite_coherence"] = (
        agg["mean_cosine"] +
        (agg["mean_bertscore_f1"] - 0.8) / 0.2 +   # rescale [0.8, 1.0] -> [0, 1]
        agg["mean_cross_encoder"]
    ) / 3

    agg.sort_values("composite_coherence", ascending=True, inplace=True)
    return agg


## Diagnostics

Prints a summary of score distributions, flagged counts, and sample
incoherent replies so you can visually verify the thresholds are reasonable.


In [ ]:
def print_diagnostics(results: pd.DataFrame, summary: pd.DataFrame) -> None:
    """Print score summaries, flag counts, and sample flagged replies."""
    print("\n" + "=" * 60)
    print("RESPONSE COHERENCE -- DIAGNOSTICS")
    print("=" * 60)
    print(f"\nTotal reply tweets scored  : {len(results):,}")
    print(f"Flagged (>=2 methods agree): {results['flag_consensus'].sum():,} "
          f"({results['flag_consensus'].mean()*100:.1f}%)\n")

    for col, thresh, label in [
        ("cosine_similarity",    COSINE_LOW_THRESHOLD,    "Cosine Similarity"),
        ("bs_f1",                BERTSCORE_LOW_THRESHOLD, "BERTScore F1"),
        ("cross_encoder_score",  CROSS_ENC_LOW_THRESHOLD, "Cross-Encoder Score"),
    ]:
        print(f"-- {label} " + "-" * (44 - len(label)))
        print(results[col].describe().round(4).to_string())
        flagged_col = f"flag_low_{col.split('_')[0]}" if col != "bs_f1" else "flag_low_bertscore"
        print(f"\nBelow threshold ({thresh}): {results[flagged_col].sum():,} replies\n")

    cols = ["thread_id", "n_replies", "composite_coherence",
            "mean_cosine", "mean_bertscore_f1", "mean_cross_encoder",
            "pct_flagged_consensus"]
    print("-- Least coherent threads (bottom 10) " + "-" * 22)
    print(summary[cols].head(10).to_string(index=False))
    print("\n-- Most coherent threads (top 10) " + "-" * 26)
    print(summary[cols].tail(10).to_string(index=False))

    print("\n-- Sample flagged replies (consensus) " + "-" * 22)
    flagged = results[results["flag_consensus"]].head(5)[
        ["id", "thread_id", "root_tweet", "tweet",
         "cosine_similarity", "bs_f1", "cross_encoder_score"]]
    for _, row in flagged.iterrows():
        print(f"\n  Thread {row['thread_id']} | Reply id {row['id']}")
        print(f"  ROOT : {row['root_tweet'][:120]}")
        print(f"  REPLY: {row['tweet'][:120]}")
        print(f"  cos={row['cosine_similarity']:.3f}  "
              f"bert_f1={row['bs_f1']:.3f}  "
              f"cross={row['cross_encoder_score']:.3f}")


## Run scoring pipeline

Switch `PAIRING_STRATEGY` to `'direct_parent'` to use direct-parent pairing
instead of thread-root pairing. Everything downstream is identical.


In [ ]:
PAIRING_STRATEGY = "thread_root"   # "thread_root" | "direct_parent"

if PAIRING_STRATEGY == "direct_parent":
    roots, replies = load_direct_parent(INPUT_CSV)
else:
    roots, replies = load_thread_root(INPUT_CSV)

# Score
replies = replies.assign(cosine_similarity=compute_cosine_similarity(replies))
replies = replies.join(compute_bertscore(replies))
replies = replies.assign(cross_encoder_score=compute_cross_encoder(replies))

# Flag and aggregate
results = flag_low_coherence(replies)
summary = summarise_by_thread(results)

print_diagnostics(results, summary)


## Save results

In [ ]:
keep_cols = [
    "id", "thread_id", "round", "user_id",
    "root_id", "root_tweet", "tweet",
    "cosine_similarity",
    "bs_precision", "bs_recall", "bs_f1",
    "cross_encoder_score",
    "flag_low_cosine", "flag_low_bertscore",
    "flag_low_crossenc", "flag_consensus",
]
results[keep_cols].to_csv(OUTPUT_RESULTS, index=False)
summary.to_csv(OUTPUT_SUMMARY, index=False)
print(f"Results -> {OUTPUT_RESULTS}")
print(f"Summary -> {OUTPUT_SUMMARY}")


## Visualisations

Four figures:
1. **Score distributions** -- histograms with KDE, threshold line, and flagged-zone shading
2. **Scatter matrix** -- pairwise correlations between the three metrics
3. **Coherence decay** -- mean score per conversation round
4. **Thread heatmap** -- composite coherence per (thread, round) cell


In [ ]:
def _add_stats_box(ax, data, color):
    """Overlay mean / median / std as a text annotation."""
    txt = f"mean  {data.mean():.3f}\nmedian {data.median():.3f}\nstd    {data.std():.3f}"
    ax.text(0.97, 0.97, txt, transform=ax.transAxes,
            ha="right", va="top", fontsize=8, color="#444441", fontfamily="monospace",
            bbox=dict(boxstyle="round,pad=0.35", fc="white", ec="#D3D1C7", lw=0.5, alpha=0.85))


def plot_histograms(results: pd.DataFrame, out_dir: str):
    """Three histograms with split colouring, KDE overlay, and threshold line."""
    fig, axes = plt.subplots(1, 3, figsize=(14, 4.5), constrained_layout=True)
    fig.suptitle("Response coherence -- score distributions", fontsize=13,
                 fontweight="bold", color="#2C2C2A", y=1.01)

    for ax, (col, (color, color_l, thresh, (lo, hi))) in zip(axes, METRIC_COLORS.items()):
        data = results[col].dropna()
        bins = np.linspace(lo, hi, 26)
        counts, edges = np.histogram(data, bins=bins)
        for left, right, count in zip(edges[:-1], edges[1:], counts):
            ax.bar(left, count, width=(right - left) * 0.92, align="edge",
                   color=color_l if right <= thresh else color, zorder=2)

        try:
            kde   = gaussian_kde(data, bw_method=0.12)
            kde_x = np.linspace(lo, hi, 300)
            ax.plot(kde_x, kde(kde_x) * len(data) * (bins[1] - bins[0]),
                    color=color, lw=2, zorder=4)
        except Exception:
            pass

        ax.axvspan(lo, thresh, color=color_l, alpha=0.15, zorder=1)
        ax.axvline(thresh, color=C_RED, lw=1.5, ls="--", zorder=5, label=f"threshold {thresh}")
        ax.axvline(data.mean(), color=color, lw=1.2, ls=":", zorder=5, label=f"mean {data.mean():.3f}")
        ax.set_xlim(lo, hi)
        ax.set_xlabel(METRIC_LABELS[col])
        ax.set_ylabel("Count" if ax is axes[0] else "")
        ax.set_title(METRIC_LABELS[col])
        _add_stats_box(ax, data, color)

        n_flagged = (data < thresh).sum()
        ax.text(0.03, 0.97, f"{n_flagged} flagged ({n_flagged/len(data)*100:.1f}%)",
                transform=ax.transAxes, ha="left", va="top", fontsize=8, color=C_RED,
                bbox=dict(boxstyle="round,pad=0.3", fc="white", ec="#F09595", lw=0.5, alpha=0.9))
        ax.legend([
            mpatches.Patch(color=color_l, label="below threshold"),
            mpatches.Patch(color=color,   label="above threshold"),
            plt.Line2D([0], [0], color=C_RED, lw=1.5, ls="--", label=f"threshold {thresh}"),
        ], loc="upper left", fontsize=7.5)

    fig.savefig(f"{out_dir}/fig1_score_distributions.png", dpi=150, bbox_inches="tight")
    plt.show()
    print(f"  [saved] fig1_score_distributions.png")


def plot_scatter_matrix(results: pd.DataFrame, out_dir: str):
    """3x3 scatter matrix coloured by consensus flag; diagonal shows per-group KDE."""
    metrics = list(METRIC_COLORS.keys())
    flagged     = results["flag_consensus"].astype(bool)
    not_flagged = ~flagged

    fig, axes = plt.subplots(3, 3, figsize=(11, 10), constrained_layout=True)
    fig.suptitle("Score correlation matrix -- coloured by consensus flag",
                 fontsize=12, fontweight="bold", color="#2C2C2A", y=1.01)

    for i, row_m in enumerate(metrics):
        for j, col_m in enumerate(metrics):
            ax = axes[i, j]
            ax.set_facecolor(C_BG)

            if i == j:
                color, _, _, (lo, hi) = METRIC_COLORS[row_m]
                xs = np.linspace(lo, hi, 300)
                for mask, c, ls in [(not_flagged, color, "-"), (flagged, C_RED, "--")]:
                    d = results.loc[mask, row_m].dropna()
                    if len(d) > 5:
                        ax.fill_between(xs, gaussian_kde(d, bw_method=0.15)(xs), color=c, alpha=0.2)
                        ax.plot(xs, gaussian_kde(d, bw_method=0.15)(xs), color=c, lw=1.5, ls=ls)
                ax.set_xlim(lo, hi)
                ax.set_title(METRIC_LABELS[row_m], fontsize=9, pad=6)
            else:
                ax.scatter(results.loc[not_flagged, col_m], results.loc[not_flagged, row_m],
                           s=6, color=C_GRAY, alpha=0.35, linewidths=0, rasterized=True)
                ax.scatter(results.loc[flagged, col_m], results.loc[flagged, row_m],
                           s=8, color=C_RED, alpha=0.65, linewidths=0, rasterized=True)
                corr = results[[col_m, row_m]].dropna().corr().iloc[0, 1]
                ax.text(0.05, 0.95, f"r = {corr:.2f}", transform=ax.transAxes, fontsize=8, va="top",
                        color="#2C2C2A", bbox=dict(boxstyle="round,pad=0.2", fc="white", ec="#D3D1C7", lw=0.5, alpha=0.85))
                ax.axvline(METRIC_COLORS[col_m][2], color=C_RED, lw=0.8, ls="--", alpha=0.6)
                ax.axhline(METRIC_COLORS[row_m][2], color=C_RED, lw=0.8, ls="--", alpha=0.6)
                lox, hix = METRIC_COLORS[col_m][3]
                loy, hiy = METRIC_COLORS[row_m][3]
                ax.set_xlim(lox, hix); ax.set_ylim(loy, hiy)

            if i == 2: ax.set_xlabel(METRIC_LABELS[col_m], fontsize=8)
            if j == 0: ax.set_ylabel(METRIC_LABELS[row_m], fontsize=8)
            ax.tick_params(labelsize=7)

    fig.legend([plt.scatter([], [], s=20, color=C_GRAY, alpha=0.6, label="coherent"),
                plt.scatter([], [], s=20, color=C_RED,  alpha=0.8, label="flagged")],
               loc="lower center", ncol=2, bbox_to_anchor=(0.5, -0.02), fontsize=9)
    fig.savefig(f"{out_dir}/fig2_score_correlations.png", dpi=150, bbox_inches="tight")
    plt.show()
    print(f"  [saved] fig2_score_correlations.png")


def plot_decay(results: pd.DataFrame, out_dir: str):
    """Mean +/- std of each metric per conversation round."""
    metrics = list(METRIC_COLORS.keys())
    agg = results.groupby("round")[metrics].agg(["mean", "std", "count"]).reset_index()
    agg.columns = ["round"] + [f"{m}_{s}" for m in metrics for s in ["mean", "std", "count"]]
    rounds = agg["round"].values

    fig, axes = plt.subplots(1, 3, figsize=(14, 4.2), constrained_layout=True)
    fig.suptitle("Coherence decay by conversation round", fontsize=13,
                 fontweight="bold", color="#2C2C2A", y=1.01)

    for ax, (col, (color, color_l, thresh, (lo, hi))) in zip(axes, METRIC_COLORS.items()):
        mu = agg[f"{col}_mean"].values
        sd = agg[f"{col}_std"].fillna(0).values
        ax.fill_between(rounds, mu - sd, mu + sd, color=color, alpha=0.12, label="+/-1 std")
        ax.plot(rounds, mu, color=color, lw=2.5, marker="o", markersize=5,
                markerfacecolor="white", markeredgecolor=color, markeredgewidth=1.5, zorder=4, label="mean")
        ax.axhline(thresh, color=C_RED, lw=1.2, ls="--", alpha=0.8, label=f"threshold {thresh}")
        if len(rounds) >= 3:
            z = np.polyfit(rounds, mu, 1)
            ax.plot(rounds, np.poly1d(z)(rounds), color=color, lw=1, ls=":", alpha=0.6,
                    label=f"trend {z[0]:+.3f}/round")
        ax.set_xlim(rounds[0] - 0.3, rounds[-1] + 0.3)
        ax.set_xlabel("Round")
        ax.set_ylabel(METRIC_LABELS[col] if ax is axes[0] else "")
        ax.set_title(METRIC_LABELS[col])
        ax.xaxis.set_major_locator(mticker.MaxNLocator(integer=True))
        ax.legend(fontsize=7.5, loc="upper right")

    fig.savefig(f"{out_dir}/fig3_coherence_decay.png", dpi=150, bbox_inches="tight")
    plt.show()
    print(f"  [saved] fig3_coherence_decay.png")


def plot_heatmap(results: pd.DataFrame, out_dir: str, max_threads: int = 50, max_rounds: int = 10):
    """Composite coherence heatmap: threads (rows) x rounds (cols)."""
    df = results.copy()
    df["bs_norm"]  = ((df["bs_f1"] - 0.7) / 0.3).clip(0, 1)
    df["composite"] = (df["cosine_similarity"] + df["bs_norm"] + df["cross_encoder_score"]) / 3

    pivot = (
        df[df["round"] <= max_rounds]
        .groupby(["thread_id", "round"])[["composite"]]
        .mean()
        .unstack("round")
    )
    pivot = pivot.loc[pivot.mean(axis=1).sort_values().index]
    if len(pivot) > max_threads:
        half = max_threads // 2
        pivot = pd.concat([pivot.head(half), pivot.tail(half)])

    fig, ax = plt.subplots(figsize=(12, max(5, len(pivot) * 0.22 + 2)), constrained_layout=True)
    fig.suptitle("Thread coherence heatmap (composite score per round)",
                 fontsize=12, fontweight="bold", color="#2C2C2A")
    sns.heatmap(pivot, ax=ax,
                cmap=sns.diverging_palette(10, 150, s=70, l=45, as_cmap=True),
                center=0.5, vmin=0.0, vmax=1.0,
                linewidths=0.3, linecolor="#EEECEA",
                cbar_kws={"label": "composite coherence", "shrink": 0.6},
                yticklabels=False)
    ax.set_xlabel("Conversation round", fontsize=10)
    ax.set_ylabel(f"Threads (n={len(pivot)}, sorted by mean coherence)", fontsize=10)

    worst_n = min(5, len(pivot) // 4)
    ax.axhline(worst_n, color=C_RED,  lw=1.2, ls="--", alpha=0.7)
    ax.axhline(len(pivot) - worst_n, color=C_TEAL, lw=1.2, ls="--", alpha=0.7)
    ax.text(-0.5, worst_n / 2, "least\ncoherent", ha="right", va="center",
            fontsize=8, color=C_RED,  transform=ax.get_yaxis_transform())
    ax.text(-0.5, len(pivot) - worst_n / 2, "most\ncoherent", ha="right", va="center",
            fontsize=8, color=C_TEAL, transform=ax.get_yaxis_transform())

    fig.savefig(f"{out_dir}/fig4_thread_heatmap.png", dpi=150, bbox_inches="tight")
    plt.show()
    print(f"  [saved] fig4_thread_heatmap.png")
    return pivot


# Run all plots
print("[viz] Generating figures ...")
plot_histograms(results, PLOT_DIR)
plot_scatter_matrix(results, PLOT_DIR)
plot_decay(results, PLOT_DIR)
pivot_data = plot_heatmap(results, PLOT_DIR)
print("[viz] All figures saved.")
